In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')

project_path = "/content/drive/MyDrive/Assignment4/Facial-Expression-Recognition"
os.makedirs(project_path, exist_ok=True)
%cd $project_path

Mounted at /content/drive
/content/drive/MyDrive/Assignment4/Facial-Expression-Recognition


In [2]:
import os

if os.path.exists("train.csv") and os.path.exists("test.csv"):
    print("Dataset files are ready.")
else:
    print("Warning: Missing train.csv or test.csv in the current folder.")

Dataset files are ready.


In [7]:
import os
import torch
import wandb
from src.models import UltimateResNet

api = wandb.Api()

run_path = "bacchia-free-university-of-tbilisi-/FER2013-Project/dv4e2sz0"
run = api.run(run_path)
wandb_config = run.config

print("Successfully synced with W&B v11 config attributes.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UltimateResNet().to(device)

local_weights = "UltimateResNet_best.pth"

if os.path.exists(local_weights):
    model.load_state_dict(torch.load(local_weights, map_location=device))
    print(f"Success! Model weights loaded cleanly from local path: {local_weights}")
else:
    print(f"Error: '{local_weights}' was not found in your current folder.")
    print("Please double check that your files are in the directory mounted in Cell 1.")

model.eval()

Successfully synced with W&B v11 config attributes.
Error: 'UltimateResNet_best.pth' was not found in your current folder.
Please double check that your files are in the directory mounted in Cell 1.


UltimateResNet(
  (prep): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (layer1): PreActResidualBlock(
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (shortcut): Sequential()
  )
  (layer2): PreActResidualBlock(
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_sta

In [8]:
import pandas as pd
import numpy as np
import torch

test_df = pd.read_csv("test.csv")

pixels = [list(map(int, p.split())) for p in test_df['pixels']]
X_test = np.array(pixels, dtype=np.float32).reshape(-1, 1, 48, 48) / 255.0
X_test_tensor = torch.tensor(X_test).to(device)

preds = []
batch_size = 128

with torch.no_grad():
    for i in range(0, len(X_test_tensor), batch_size):
        batch = X_test_tensor[i:i+batch_size]

        out_orig = model(batch)
        out_flip = model(torch.flip(batch, dims=[-1]))
        out_shift = model(torch.roll(batch, shifts=(2, 2), dims=(-2, -1)))

        final_out = (out_orig + out_flip + out_shift) / 3.0
        batch_preds = final_out.argmax(dim=1)
        preds.extend(batch_preds.cpu().numpy())

submission_df = pd.DataFrame({
    "ID": test_df["ID"] if "ID" in test_df.columns else test_df.index,
    "Label": preds
})

submission_df.to_csv("submission.csv", index=False)
print("Saved predictions to submission.csv.")

Saved predictions to submission.csv.
